# Feature Engineering

This notebook will create n-grams to be used as phrase-enhanced tokens, perform NER, normalize extracted entities, conduct TF-IDF, and create a CountVEctorizer using phrase-enhanced tokens to create a document-term count matrix.

## Set up

In [ ]:
from pathlib import Path
import re
import unicodedata
from collections import Counter

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import (
    ENGLISH_STOP_WORDS,
    TfidfVectorizer,
    CountVectorizer
)

from scipy.sparse import save_npz

# Gensim phrase detection
from gensim.models.phrases import (
    Phrases,
    Phraser,
    ENGLISH_CONNECTOR_WORDS
)

import joblib

# Project paths
PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Processed data directory:", PROCESSED_DIR)

In [ ]:
# Load preprocessed corpus
input_path = PROCESSED_DIR / "ungdc_preprocessed.parquet"

metadata = pd.read_parquet(input_path)

print(f"Loaded {len(metadata):,} speeches")
print(f"Countries/entities: {metadata['country_code'].nunique():,}")
print(f"Year range: {metadata['year'].min()}–{metadata['year'].max()}")

# Confirm that the columns needed for this notebook are available
required_columns = [
    "filename",
    "country_code",
    "year",
    "raw_text",
    "tokens"
]

missing_columns = [
    col for col in required_columns
    if col not in metadata.columns
]

assert not missing_columns, f"Missing columns: {missing_columns}"

display(metadata.head())

In [ ]:
# Recreate stopword list
stopwords = set(ENGLISH_STOP_WORDS)

# Preserve negation because it can change the meaning of a statement
stopwords -= {"no", "nor", "not", "never", "neither"}

# Generic diplomatic/procedural terms
un_procedural_stopwords = {
    "mr",
    "mrs",
    "madam",
    "chairman",
    "chairperson",
    "delegation",
    "delegate",
    "delegates",
    "session",
    "plenary"
}

stopwords.update(un_procedural_stopwords)

# Ceremonial terms that are generally less useful for thematic analysis
ceremonial_candidates = {
    "congratulations",
    "congratulate",
    "pleasure",
    "appreciation",
    "behalf",
    "occasion",
    "thanks",
    "wish"
}

stopwords.update(ceremonial_candidates)

# Explicitly retain these potentially meaningful terms
stopwords -= {
    "assembly",
    "secretary",
    "president"
}

print(f"Total stopwords: {len(stopwords):,}")

In [ ]:
# Tokenize while preserving stopwords
def tokenize_for_phrases(text):
    """
    Normalize and tokenize text while preserving stopwords.
    Used specifically for bigram and trigram detection.
    """

    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Preserve dotted acronyms:
    # U.S. -> u_s, U.N. -> u_n, U.S.A. -> u_s_a
    text = re.sub(
        r"\b(?:[a-z]\.){2,}",
        lambda m: m.group(0).rstrip(".").replace(".", "_"),
        text
    )

    # Extract words and normalized acronyms
    return re.findall(
        r"[a-z]+(?:_[a-z]+)*(?:'[a-z]+)?",
        text
    )


metadata["phrase_tokens"] = metadata["raw_text"].fillna("").apply(
    tokenize_for_phrases
)

print("Created stopword-preserving tokens.")

display(
    metadata[["filename", "phrase_tokens"]].head(5)
)

## Detect n-grams

In [ ]:
# First detect bigrams
# Each speech is treated as a separate sequence of tokens
sentences = metadata["phrase_tokens"].tolist()

# Train the bigram model
bigram_model = Phrases(
    sentences,
    min_count=5,
    threshold=10,
    connector_words=ENGLISH_CONNECTOR_WORDS,
    delimiter="__"
)

# Convert to a more memory-efficient model for applying phrases
bigram_phraser = Phraser(bigram_model)

print("Bigram model trained.")

In [ ]:
# Inspect detected bigrams
bigram_counts = Counter()
bigram_documents = Counter()

for tokens in sentences:
    transformed = bigram_phraser[tokens]

    # Double underscores distinguish learned phrases from dotted acronyms
    phrases = [token for token in transformed if "__" in token]

    bigram_counts.update(phrases)
    bigram_documents.update(set(phrases))


top_bigrams = pd.DataFrame(
    [
        {
            "bigram": phrase.replace("__", " "),
            "frequency": count,
            "document_frequency": bigram_documents[phrase]
        }
        for phrase, count in bigram_counts.most_common(50)
    ]
)

display(top_bigrams)

In [ ]:
# Now detect trigrams
# Train the trigram model on bigram-transformed speeches
trigram_model = Phrases(
    (bigram_phraser[sentence] for sentence in sentences),
    min_count=5,
    threshold=10,
    connector_words=ENGLISH_CONNECTOR_WORDS,
    delimiter="__"
)

trigram_phraser = Phraser(trigram_model)

print("Trigram model trained.")

## Create phrase models and analysis tokens

In [ ]:
def create_analysis_tokens(tokens):
    """
    Apply bigram and trigram detection, then remove standalone stopwords.
    """

    # Apply bigram detection
    tokens = bigram_phraser[tokens]

    # Apply trigram detection
    tokens = trigram_phraser[tokens]

    # Remove standalone stopwords and one-character tokens
    tokens = [
        token for token in tokens
        if token not in stopwords
        and len(token.replace("_", "")) > 1
    ]

    return tokens


metadata["analysis_tokens"] = metadata["phrase_tokens"].apply(
    create_analysis_tokens
)

metadata["analysis_text"] = metadata["analysis_tokens"].apply(
    lambda tokens: " ".join(tokens)
)

display(
    metadata[
        ["filename", "tokens", "analysis_tokens"]
    ].head(5)
)

In [ ]:
# Inspect resulting phrases
phrase_counts = Counter()
phrase_documents = Counter()

for tokens in metadata["analysis_tokens"]:
    phrases = [token for token in tokens if "__" in token]

    phrase_counts.update(phrases)
    phrase_documents.update(set(phrases))


top_phrases = pd.DataFrame(
    [
        {
            "phrase": phrase.replace("__", " "),
            "frequency": count,
            "document_frequency": phrase_documents[phrase]
        }
        for phrase, count in phrase_counts.most_common(50)
    ]
)

display(top_phrases)

## TF-IDF

In [ ]:
# Create TF-IDF matrix
tfidf_vectorizer = TfidfVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=5,
    max_df=0.80
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    metadata["analysis_text"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Number of TF-IDF features:", len(tfidf_vectorizer.vocabulary_))

## CountVectorizer

In [ ]:
# Create count-based document-term matrix (useful for LDA)
count_vectorizer = CountVectorizer(
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    lowercase=False,
    min_df=5,
    max_df=0.80
)

count_matrix = count_vectorizer.fit_transform(
    metadata["analysis_text"]
)

print("Count matrix shape:", count_matrix.shape)
print("Number of count features:", len(count_vectorizer.vocabulary_))

## Save Outputs

In [ ]:
# Save document-level metadata and transformed text
feature_columns = [
    "filename",
    "country_code",
    "session",
    "year",
    "analysis_text",
    "analysis_tokens"
]

feature_data = metadata[feature_columns].copy()

feature_data.to_parquet(
    PROCESSED_DIR / "ungdc_features.parquet",
    index=False
)

# Save sparse document-term matrices
save_npz(
    PROCESSED_DIR / "ungdc_tfidf_matrix.npz",
    tfidf_matrix
)

save_npz(
    PROCESSED_DIR / "ungdc_count_matrix.npz",
    count_matrix
)

# Save fitted vectorizers so we can recover feature names later
joblib.dump(
    tfidf_vectorizer,
    PROCESSED_DIR / "tfidf_vectorizer.joblib"
)

joblib.dump(
    count_vectorizer,
    PROCESSED_DIR / "count_vectorizer.joblib"
)

# Save phrase models for reuse
joblib.dump(
    bigram_phraser,
    PROCESSED_DIR / "bigram_phraser.joblib"
)

joblib.dump(
    trigram_phraser,
    PROCESSED_DIR / "trigram_phraser.joblib"
)

print("Saved feature data and models to:", PROCESSED_DIR)

## NER

In [ ]:
import spacy

# Disable components not needed for named entity recognition
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "tagger", "lemmatizer", "attribute_ruler"]
)

print("Loaded spaCy model:", nlp.meta["name"])

In [ ]:
# Extract named entities
entity_labels = {
    "GPE",       # Countries, cities, geopolitical entities
    "LOC",       # Geographic locations
    "ORG",       # Organizations and institutions
    "PERSON",    # People
    "NORP"       # Nationalities, religious and political groups
}

entity_records = []

texts = (
    (text if isinstance(text, str) else "", i)
    for i, text in enumerate(metadata["raw_text"])
)

for count, (doc, i) in enumerate(
    nlp.pipe(texts, as_tuples=True, batch_size=64),
    start=1
):
    row = metadata.iloc[i]

    for ent in doc.ents:
        if ent.label_ in entity_labels:
            entity_records.append({
                "filename": row["filename"],
                "country_code": row["country_code"],
                "year": row["year"],
                "entity_text": ent.text,
                "entity_label": ent.label_,
                "start_char": ent.start_char,
                "end_char": ent.end_char
            })

    if count % 250 == 0 or count == len(metadata):
        print(f"Processed {count:,}/{len(metadata):,} documents "
              f"({count / len(metadata) * 100:.1f}%)")

entities = pd.DataFrame(entity_records)

print(f"\nExtracted {len(entities):,} entity mentions")
print(f"Unique entity strings: {entities['entity_text'].nunique():,}")

In [ ]:
# Inspect entity categories
display(
    entities["entity_label"]
    .value_counts()
    .rename_axis("entity_type")
    .reset_index(name="mentions")
)

In [ ]:
# Explore most frequently mentioned entities
entity_summary = (
    entities
    .groupby(["entity_text", "entity_label"])
    .agg(
        mentions=("entity_text", "size"),
        speeches=("filename", "nunique")
    )
    .reset_index()
    .sort_values("mentions", ascending=False)
)

display(entity_summary.head(50))

In [ ]:
# Save NER results
entities.to_parquet(
    PROCESSED_DIR / "ungdc_entities.parquet",
    index=False
)

entity_summary.to_parquet(
    PROCESSED_DIR / "ungdc_entity_summary.parquet",
    index=False
)

print("Saved NER outputs to:", PROCESSED_DIR)